In [2]:
import math
import numpy as np
import matplotlib.pyplot as plt
from pyfluids import Fluid, FluidsList, Input

# have to be modify is temperature dependent cp and gamma

# ---------------------------------------------------------
# 1. INPUT CONSTANTS (Shared by both stages)
# ---------------------------------------------------------
T_turbine_inlet = 1630   # K
P_turbine_inlet = 518.47 # kPa

P_turbine_outlet = 182 # kPa
T_turbine_outlet = 1303.29   # K

T_avg = ( T_turbine_inlet + T_turbine_outlet )/2
P_avg = ( P_turbine_inlet + P_turbine_outlet )/2

# define air properties at T_avg and P_avg
air_mean = Fluid(FluidsList.Air).with_state(Input.temperature(T_avg), Input.pressure(P_avg))

cp = air_mean.specific_heat
heat_ratio = 1.3  # assume gamma = 1.3 for this temperature range
w41 = 94.39           
n_stages = 2
omega_tubrine = 166.29 # rotational speed of turbine (rev/s)

# Design coefficients
work_coeff = 0.8
flow_coeff = 0.55
stage_efficiency = 0.93 # obtained from Smith's Chart
m_bar = 1.28         # assumed chocked stator exit M = 1 

# ---------------------------------------------------------
# 2. FUNCTIONS
# ---------------------------------------------------------
def reaction(v_axial, U, alpha_2, alpha_3):
    return 1 - v_axial / (2 * U) * (math.tan(math.radians(alpha_2)) + math.tan(math.radians(alpha_3)))

def alpha3_to_alpha2(work_coeff, flow_coeff, alpha_3):
    return math.degrees(math.atan(work_coeff / flow_coeff + math.tan(math.radians(alpha_3))))

def alpha_to_beta(flow_coeff, alpha):
    return math.degrees(math.atan(math.tan(math.radians(alpha)) - 1 / flow_coeff))

def annulus_area(m_bar, mass_flow, P0, T0, cp):
    return mass_flow * math.sqrt(cp * T0) / (P0 * m_bar * 1000)

def blade_height(A, r, alpha2):
    return A / (2 * math.pi * r * math.cos(math.radians(alpha2)))

# ---------------------------------------------------------
# 3. STAGE 1 CALCULATIONS
# ---------------------------------------------------------
# Initial conditions
T_in_s1 = T_turbine_inlet
P_in_s1 = P_turbine_inlet
dh_stage = (cp * (T_turbine_inlet- T_turbine_outlet)) / n_stages 

# tagential speed at mean radius m/s
U_mean = math.sqrt(dh_stage/work_coeff)  

# mean radius m
r_mean = U_mean / (omega_tubrine * 2 * math.pi)

# axial velocity
v_axial = flow_coeff * U_mean 
# Angles Stage 1
alpha_mean_1_s1 = 0
alpha_mean_3_s1 = -30 # can change 
alpha_mean_2_s1 = alpha3_to_alpha2(work_coeff, flow_coeff, alpha_mean_3_s1)

# Relative angles for Stage 1
beta_mean_2_s1 = alpha_to_beta(flow_coeff, alpha_mean_2_s1)
beta_mean_3_s1 = alpha_to_beta(flow_coeff, alpha_mean_3_s1)

# Turning Angles for stage 2
turning_stator_s1 = abs(alpha_mean_1_s1 - alpha_mean_2_s1)
turning_rotor_s1 = abs(beta_mean_2_s1 - beta_mean_3_s1)

# Thermo Stage 1
T_out_s1 = T_in_s1 - dh_stage / cp
T_out_isen_s1 = T_in_s1 - (T_in_s1 - T_out_s1) / stage_efficiency
P_exit_s1 = P_in_s1 * (T_out_isen_s1 / T_in_s1)**(heat_ratio / (heat_ratio - 1))
pr_s1 = P_in_s1 / P_exit_s1
ttt_efficiency_s1 = (1 - T_out_s1 / T_in_s1)/(1- (P_exit_s1/P_in_s1)**((heat_ratio - 1)/heat_ratio))


# Geometry & Reaction Stage 1
A_mean_s1 = annulus_area(m_bar, w41, P_in_s1, T_in_s1, cp)
h_mean_s1 = blade_height(A_mean_s1, r_mean, alpha_mean_2_s1)
r_hub_s1 = r_mean - h_mean_s1/2
r_tip_s1 = r_hub_s1 + h_mean_s1
R_mean_s1 = reaction(v_axial, U_mean, alpha_mean_2_s1, alpha_mean_3_s1)

# ---------------------------------------------------------
# 4. STAGE 2 CALCULATIONS
# ---------------------------------------------------------
# Hand-off from Stage 1
T_in_s2 = T_out_s1   
P_in_s2 = P_exit_s1  

# New Angle Definitions for Stage 2
alpha_mean_1_s2 = alpha_mean_3_s1  # Inherits swirl from S1 exit (-30 deg)
alpha_mean_3_s2 = 0                # Axial exit for Stage 2

# Recalculate alpha2 for Stage 2 (based on alpha3 = 0)
alpha_mean_2_s2 = alpha3_to_alpha2(work_coeff, flow_coeff, alpha_mean_3_s2)

# Relative angles for Stage 2
beta_mean_2_s2 = alpha_to_beta(flow_coeff, alpha_mean_2_s2)
beta_mean_3_s2 = alpha_to_beta(flow_coeff, alpha_mean_3_s2)

# Turning Angles for stage 2
turning_stator_s2 = abs(alpha_mean_1_s2 - alpha_mean_2_s2)
turning_rotor_s2 = abs(beta_mean_2_s2 - beta_mean_3_s2)

# Thermo Stage 2
T_out_s2 = T_in_s2 - dh_stage / cp
T_out_isen_s2 = T_in_s2 - (T_in_s2 - T_out_s2) / stage_efficiency
P_exit_s2 = P_in_s2 * (T_out_isen_s2 / T_in_s2)**(heat_ratio / (heat_ratio - 1))
pr_s2 = P_in_s2 / P_exit_s2
ttt_efficiency_s2 = (1 - T_out_s2 / T_in_s2)/(1- (P_exit_s2/P_in_s2)**((heat_ratio - 1)/heat_ratio))

# Geometry & Reaction Stage 2
A_mean_s2 = annulus_area(m_bar, w41, P_in_s2, T_in_s2, cp)
h_mean_s2 = blade_height(A_mean_s2, r_mean, alpha_mean_2_s2)
r_hub_s2 = r_mean - h_mean_s2/2
r_tip_s2 = r_hub_s2 + h_mean_s2
R_mean_s2 = reaction(v_axial, U_mean, alpha_mean_2_s2, alpha_mean_3_s2)

# ---------------------------------------------------------
# 5. OUTPUT
# ---------------------------------------------------------
print(f"No. of stages: {n_stages}")
print(f"Work Coefficient: {work_coeff}")
print(f"Flow Coefficient: {flow_coeff}")
print(f"Absolute flow angle at stage 1 rotor outlet: {alpha_mean_3_s1:.2f}")

print("-" * 50)
print(f"Mean Radius {r_mean:.2f} m")
print(f"U_mean: {U_mean:.2f} m/s")
print(f"Specific Heat at constant pressure (Cp) at Tavg = {T_avg:.2f}K : {cp:.2f} J/kg*k")


print("-" * 50)
print(f"{'PARAMETER':<25} | {'STAGE 1':<10} | {'STAGE 2':<10}")
print("-" * 50)
print(f"{'Alpha 1 (Inlet)':<25} | {alpha_mean_1_s1:<10.2f} | {alpha_mean_1_s2:<10.2f}")
print(f"{'Alpha 2 (Rotor In)':<25} | {alpha_mean_2_s1:<10.2f} | {alpha_mean_2_s2:<10.2f}")
print(f"{'Alpha 3 (Exit)':<25} | {alpha_mean_3_s1:<10.2f} | {alpha_mean_3_s2:<10.2f}")
print("-" * 50)
print(f"{'Beta 2 (Rotor In)':<25} | {beta_mean_2_s1:<10.2f} | {beta_mean_2_s2:<10.2f}")
print(f"{'Beta 3 (Exit)':<25} | {beta_mean_3_s1:<10.2f} | {beta_mean_3_s2:<10.2f}")
print("-" * 50)
print(f"{'Stator Turning Angles':<25} | {turning_stator_s1:<10.2f} | {turning_stator_s2:<10.2f}")
print(f"{'Rotor Turning Angles':<25} | {turning_rotor_s1:<10.2f} | {turning_rotor_s2:<10.2f}")
print("-" * 50)
print(f"{'Degree of Reaction':<25} | {R_mean_s1:<10.3f} | {R_mean_s2:<10.3f}")
print(f"{'Stage Pressure Ratio':<25} | {pr_s1:<10.3f} | {pr_s2:<10.3f}")
print(f"{'Total to total efficiency':<25} | {ttt_efficiency_s1:<10.3f} | {ttt_efficiency_s2:<10.3f}")
print("-" * 50)
print(f"{'Blade Height (m)':<25} | {h_mean_s1:<10.4f} | {h_mean_s2:<10.4f}")
print(f"{'Tip Radius (m)':<25} | {r_tip_s1:<10.4f} | {r_tip_s2:<10.4f}")
print(f"{'Hub Radius (m)':<25} | {r_hub_s1:<10.4f} | {r_hub_s2:<10.4f}")
print(f"{'Exit Pressure (kPa)':<25} | {P_exit_s1:<10.2f} | {P_exit_s2:<10.2f}")
print(f"{'Exit Temperature (K)':<25} | {T_out_s1:<10.2f} | {T_out_s2:<10.2f}")
print(f"{'Inlet Temperature (K)':<25} | {T_in_s1:<10.2f} | {T_in_s2:<10.2f}")

No. of stages: 2
Work Coefficient: 0.8
Flow Coefficient: 0.55
Absolute flow angle at stage 1 rotor outlet: -30.00
--------------------------------------------------
Mean Radius 0.48 m
U_mean: 501.60 m/s
Specific Heat at constant pressure (Cp) at Tavg = 1466.64K : 1232.19 J/kg*k
--------------------------------------------------
PARAMETER                 | STAGE 1    | STAGE 2   
--------------------------------------------------
Alpha 1 (Inlet)           | 0.00       | -30.00    
Alpha 2 (Rotor In)        | 41.26      | 55.49     
Alpha 3 (Exit)            | -30.00     | 0.00      
--------------------------------------------------
Beta 2 (Rotor In)         | -43.26     | -19.98    
Beta 3 (Exit)             | -67.34     | -61.19    
--------------------------------------------------
Stator Turning Angles     | 41.26      | 85.49     
Rotor Turning Angles      | 24.08      | 41.21     
--------------------------------------------------
Degree of Reaction        | 0.918      | 0.600    

In [3]:
T_turbine_inlet = 1630   # K
P_turbine_inlet = 518.47 # kPa

P_turbine_outlet = 1412.19 # kPa
T_turbine_outlet = 1303.29   # K

T_avg = ( T_turbine_inlet + T_turbine_outlet )/2
P_avg = ( P_turbine_inlet + P_turbine_outlet )/2

air_mean = Fluid(FluidsList.Air).with_state(
    Input.temperature(T_avg), 
    Input.pressure(P_avg)
)

air_mean.specific_heat

1232.1855858000542

In [5]:
dh_stage /1000



201.2836539853436